## LAMP ML Target design 

first iteration of target design: Only doing SNPs, main variables are the mutation rate per nucleotide and the mutation density (ie spread evenly or clustered) 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import date
from Bio import AlignIO

In [ ]:
#import WT sequence
seq_file_path  = "/Users/zlevine/Documents/Sabeti_Lab/LAMP_ML/Final_designs_3-13/WT_target_sequence_10blocks_350len_SR1.7_2025-03-13.csv"
WT_seq = pd.read_csv(seq_file_path,index_col = 0).loc["WT_1","sequence"]
print(len(WT_seq))

In [ ]:
# mutation_matrix = {
# 'A': {'G': 0.75, 'C': 0.125, 'T': 0.125},
# 'G': {'A': 0.75, 'C': 0.125, 'T': 0.125},
# 'C': {'A': 0.125, 'G': 0.125, 'T': 0.75},
# 'T': {'A': 0.125, 'C': 0.75, 'G': 0.125}
# }

# nucleotides = "ACGT"
# sequence = list(WT_seq)  # Convert string to list to allow mutation

# # Determine positions eligible for mutation based on rate 'u'
# mutant_positions = [i for i in range(len(sequence)) if random.random() < u]

# trans_count = 0
# tranv_count = 0
# for pos in range(len(WT_seq)):
#     original_nucleotide = sequence[pos]
#     # Choose a new nucleotide different from the original one
#     possible_nucleotides = list(mutation_matrix[original_nucleotide].keys())
#     probabilities = list(mutation_matrix[original_nucleotide].values())

#     new_nucleotide = random.choices(possible_nucleotides, weights=probabilities)[0]

#     #new_nucleotide = random.choice([nuc for nuc in nucleotides if nuc != original_nucleotide])
#     sequence[pos] = new_nucleotide
    
#     if original_nucleotide in ['A','G']: 
#         if new_nucleotide in ['A','G']: 
#             trans_count +=1 
#         else :
#             tranv_count +=1 
#     if original_nucleotide in ['C','T']: 
#         if new_nucleotide in ['C','T']: 
#             trans_count +=1 
#         else :
#             tranv_count +=1 
    

# Add diversity to targets

### Define functions

In [ ]:
import numpy as np
import random
import matplotlib.pyplot as plt

def mutate_sequence(sequence, u, d):
    """
    Mutates a given DNA sequence based on a custom model.

    Parameters:
    - sequence (str): The target DNA sequence to mutate.
    - u (float): Overall rate of mutations per nucleotide. A value between 0 and 1.
    - d (float): Density of mutations. Affects the clustering of mutations.

    Returns:
    - mutated_sequence (str): The resulting DNA sequence after mutations.
    - mutant_positions (list): The list of positions where mutations occurred.
    """
    
    mutation_matrix = {
    'A': {'G': 0.75, 'C': 0.125, 'T': 0.125},
    'G': {'A': 0.75, 'C': 0.125, 'T': 0.125},
    'C': {'A': 0.125, 'G': 0.125, 'T': 0.75},
    'T': {'A': 0.125, 'C': 0.75, 'G': 0.125}
}

    nucleotides = "ACGT"
    sequence = list(sequence)  # Convert string to list to allow mutation

    # Determine positions eligible for mutation based on rate 'u'
    mutant_positions = [i for i in range(len(sequence)) if random.random() < u]

    # Adjust the list of mutant positions based on mutation density 'd'
    if d < 1:
        selection_probability = d 
        mutant_positions = [pos for pos in mutant_positions if random.random() < selection_probability]

    for pos in mutant_positions:
        original_nucleotide = sequence[pos]
        # Choose a new nucleotide different from the original one
        possible_nucleotides = list(mutation_matrix[original_nucleotide].keys())
        probabilities = list(mutation_matrix[original_nucleotide].values())

        new_nucleotide = random.choices(possible_nucleotides, weights=probabilities)[0]

        #new_nucleotide = random.choice([nuc for nuc in nucleotides if nuc != original_nucleotide])
        sequence[pos] = new_nucleotide

    return ''.join(sequence), mutant_positions

def plot_sequence_with_mutations(sequence, mutant_positions, ax, title = "Sequence with Mutations"):
#     """
#     Plots the target sequence as a horizontal line with mutations as vertical lines.

#     Parameters:
#     - sequence (str): The DNA sequence.
#     - mutant_positions (list): The list of positions where mutations occurred.
#     """
#     #plt.figure(figsize=(12, 1))

#     # Plot the target sequence as a horizontal black line
#     ax.hlines(0, 0, len(sequence) - 1, colors='black', lw=2)

#     # Plot the mutant positions as vertical red lines
#     for pos in mutant_positions:
#         ax.vlines(pos, -0.5, 0.5, colors='red', lw=2)

#     # Add labels and finalize the plot
#     ax.set_xlabel("Position in Sequence")
#     ax.set_yticks([])
#     ax.set_title(title)
#     ax.set_xlim(-1, len(sequence))
#     #plt.show()

    #sameed version
    """
    Plots the target sequence as a horizontal line with mutations as single-pixel vertical lines.
    """
    # Clear the axis first to ensure clean plotting
    ax.clear()
    
    # Create a blank white image the length of the sequence
    img_height = 50  # Height in pixels
    img_width = len(sequence)  # Width in pixels (1 pixel per base)
    img = np.ones((img_height, img_width, 3))  # White RGB background
    
    # Draw a horizontal black line in the middle
    middle_y = img_height // 2
    img[middle_y, :, :] = 0  # Black RGB
    
    # Mark mutations with single red pixel vertical lines
    for pos in mutant_positions:
        if 0 <= pos < img_width:  # Ensure position is within bounds
            # Draw a vertical red line at the mutation position
            img[middle_y-10:middle_y+10, pos, 0] = 1  # Red channel
            img[middle_y-10:middle_y+10, pos, 1:3] = 0  # Green and Blue channels
    
    # Display the image
    ax.imshow(img, aspect='auto', interpolation='nearest')
    ax.set_yticks([])
    ax.set_xticks([0, len(sequence)//4, len(sequence)//2, 3*len(sequence)//4, len(sequence)-1])
    ax.set_title(title,fontsize = 10)
    
    # Remove axis spines for cleaner look
    for spine in ax.spines.values():
        spine.set_visible(False)

In [ ]:
def mutate_sequence_fixed_count(sequence, u, d):
    """
    Mutates a given sequence with a fixed number of mutations, influenced by density.

    Params:
    - sequence (str): Original DNA sequence.
    - u (float): Mutation rate, determining the total number of mutations.
    - d (float): Density factor influencing clustering.

    Returns:
    - mutated_sequence (str): The mutated sequence.
    - mutation_positions (list): Positions where mutations occur.
    """
    nucleotides = "ACGT"
    sequence = list(sequence)
    
    # Determine the number of mutations based on the sequence length and mutation rate
    total_mutations = int(len(sequence) * u)

    # Select positions to mutate
    all_positions = list(range(len(sequence)))
    
    if len(all_positions) < total_mutations:
        raise ValueError("Mutation rate is too high for sequence length!")

    # Adjust the spread or clustering of mutations
    # Shuffle the positions at different levels of clustering
    if d < 0.5:
        # Lower density, more clustering -> pick a smaller random subset first
        cluster_center_count = max(int(total_mutations * (1 - d)), 1)
        cluster_centers = random.sample(all_positions, cluster_center_count)
        mutation_positions = []
        for center in cluster_centers:
            # Add nearby positions up to total_mutations
            start = max(0, center - int(0.1 * len(sequence)))
            end = min(len(sequence), center + int(0.1 * len(sequence)))
            nearby_positions = list(range(start, end))
            mutation_positions.extend(random.sample(nearby_positions, min(total_mutations, len(nearby_positions))))
            total_mutations -= len(nearby_positions)
            if total_mutations <= 0:
                break
    else:
        # Higher density, more spread -> sample uniformly
        mutation_positions = random.sample(all_positions, total_mutations)

    for pos in mutation_positions:
        original_nucleotide = sequence[pos]
        new_nucleotide = random.choice([nuc for nuc in nucleotides if nuc != original_nucleotide])
        sequence[pos] = new_nucleotide

    return ''.join(sequence), mutation_positions

### Generate targets and save to fasta 

In [ ]:
len_block = 350
n_blocks = 10
block_bounds = {f'block_{i}':(i*len_block, i*len_block + len_block) for i in range(n_blocks)}
block_bounds

In [ ]:
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO

#Example input data
sequences = [WT_seq]
record_ids = ["T00"]
record_descriptions = ["u_0_d_0"]
target_n = 1
seq_records = [SeqRecord(WT_seq, id="T00", description="u_0_d_0")]

#create a figure to plot the sequences 
#mutation_rates = [0.01,0.03,0.04,0.05,0.06,0.07,0.1]
mutation_rates = [0.015,0.030,0.049,0.0594,0.072,0.085,0.105]
mutation_densities = [0.5,0.8,1]

fig, axes = plt.subplots(21,1, figsize = (8,12)) 
fig.tight_layout(h_pad = 4)
#make sequences with different mutation frequencies and mutation densities  
for u in mutation_rates:
    for d in mutation_densities:
        this_seq = ""
        this_positions = []
        #generate the mutant sequence separately for each block because we want the mutation density function applied to each block individual 
        for block, brange in block_bounds.items() :
            target_sequence = WT_seq[brange[0]:brange[1]]
        
            mutated_sequence, mutation_positions = mutate_sequence_fixed_count(target_sequence, u, d)
            this_seq += mutated_sequence
            this_positions += [m+brange[0] for m in mutation_positions]
            
            
        #sava e it 
        sequences.append(this_seq)
        record_descriptions.append(f"u_{u:.3f}_d_{d:.1f}")
        
       
        if target_n < 10 :
            tname = f"T0{target_n}"
            seq_records.append(SeqRecord(this_seq, id=f"T0{target_n}", description=f"u_{u:.3f}_d_{d:.1f}"))


        else : 
            tname = f"T{target_n}"
            seq_records.append(SeqRecord(this_seq, id=f"T{target_n}", description=f"u_{u:.3f}_d_{d:.1f}"))

        record_ids.append(tname)

        #plot it 
        plot_sequence_with_mutations(this_seq, this_positions, title = f"{tname} : u={u:.3f} d={d:.1f}",
                                    ax = axes[target_n -1])


        target_n +=1 


In [ ]:
# # Create SeqRecord objects
seq_records = []
for sequence, record_id, description in zip(sequences, record_ids, record_descriptions):
    # Create a Seq object
    seq = Seq(sequence)

    # Create a SeqRecord object
    seq_record = SeqRecord(seq, id=record_id, description=description)

    # Add to list
    seq_records.append(seq_record)

# Optionally, write the SeqRecord objects to a file, e.g., in FASTA format
output_file = f"/Users/zlevine/Documents/Sabeti_Lab/LAMP_ML/mut_targ_testing/{str(date.today())}_mut_sequences.fasta"
SeqIO.write(seq_records, output_file, "fasta")

### Plot characteristics 

In [ ]:
#count mismatches per primer for all sets 
primers = pd.read_csv("/Users/zlevine/Documents/Sabeti_Lab/LAMP_ML/mut_targ_testing/4-21_primer_bind.csv",index_col = 0).to_dict(orient = 'index')
pd.DataFrame.from_dict(primers, orient = 'index')

In [ ]:
primers_df = pd.DataFrame.from_dict(primers, orient = 'index')
len(set(primers_df.primer_set)) * 6

In [ ]:
def count_mismatches(seq1, seq2):
    # Mapping of degenerate bases
    degenerate_map = {
        'A': {'A'}, 'C': {'C'}, 'G': {'G'}, 'T': {'T'},
        'R': {'A', 'G'}, 'Y': {'C', 'T'}, 'S': {'G', 'C'},
        'W': {'A', 'T'}, 'K': {'G', 'T'}, 'M': {'A', 'C'},
        'B': {'C', 'G', 'T'}, 'D': {'A', 'G', 'T'}, 'H': {'A', 'C', 'T'},
        'V': {'A', 'C', 'G'}, 'N': {'A', 'C', 'G', 'T'}
    }

    # Check if sequences are equal length
    if len(seq1) != len(seq2):
        print("Error! Sequences are of unequal lengths.")
        return None

    # Initialize mismatch counter
    mismatches = 0

    # Compare sequences
    for base1, base2 in zip(seq1, seq2):
        # Get the set of possible matches for each base
        set1 = degenerate_map.get(base1, {base1})
        set2 = degenerate_map.get(base2, {base2})
        
        # Check if there's no overlap in possible matches
        if set1.isdisjoint(set2):
            mismatches += 1

    return mismatches

In [ ]:
from Bio import SeqIO
from Bio.Seq import Seq

def analyze_primers(msa_file, primers):
    """
    Analyzes primers against sequences in an MSA file.
    
    Parameters:
    - msa_file (str): Path to the FASTA file containing the MSA.
    - primers (dict): Dictionary with primer names as keys and dictionaries as values.
                      Each dictionary should contain:
                      - 'bind_start': int, starting index of binding region.
                      - 'bind_end': int, ending index of binding region (exclusive).
                      - 'bind_direction': str, either 'forward' or 'reverse'.
                      - 'bind_seq': str, binding sequence to compare against.
                      
    Returns:
    - dict: A dictionary with primer names as keys and lists of results for each sequence.
    """
    results = []
    
    # Read sequences from the MSA file
    sequences = list(SeqIO.parse(msa_file, "fasta"))
    
    for primer_name, primer_info in primers.items():
        primer_results = []
        
        direction = primer_info['direction']
        
        bind_start = primer_info['start'] #bc geneious is 1 indexed 
        bind_end = primer_info['end']
        
        primer_seq = primer_info['sequence']
        primer_type = primer_info['primer_type']

        for record in sequences:
    
            target_seq = str(record.seq)
            
            # Determine the binding region
            bind_region = target_seq[bind_start:bind_end]
            
            if len(bind_region.replace("N","")) != len(primer_seq) :
                print(f"Error! Primer and target seq are different lengths for {record.id} {primer_name}") 
            else :
                
                if direction == 'RC':
                    # Assuming you want to handle reverse sequence where needed
                    bind_seq = str(Seq(primer_seq).reverse_complement())
                elif direction == 'F' :
                    bind_seq = primer_seq
                else : 
                    print(f"error! missing direction for {primer_name}")

                # Calculate percentage of unambiguous bases
                unambiguous_bases = sum(1 for base in bind_region if base != 'N')
                percentage_unambiguous = (unambiguous_bases / len(bind_region)) * 100

                # Sum mismatches and store positions 
                mm_count = count_mismatches(bind_region, bind_seq) 
                    # if base1 != base2 :
                    #     mm_count += 1 
                    #     mms.append(f"{base2}{i+1}{base1}")

               # Append result as a dict to the results list
                results.append({
                    'record': record.id,
                    'record_description':record.description,
                    'primer_name': primer_name,
                    'primer_type': primer_type,
                    'primer_set': primer_info['primer_set'],
                    'primer_direction':primer_info['direction'],
                    'percentage_unambiguous': percentage_unambiguous,
                    'mismatch_count': mm_count,
                    'target_seq':bind_region,
                    'primer_seq':primer_info['sequence'],
                    'bind_seq':bind_seq
                })
    
    # Create a DataFrame from the results list
    df_results = pd.DataFrame(results)
    return df_results



msa_file =  '/Users/zlevine/Documents/Sabeti_Lab/LAMP_ML/final_mutant_target_designs/2025-06-04_mut_sequences.fasta' #f"/Users/zlevine/Documents/Sabeti_Lab/LAMP_ML/mut_targ_testing/{str(date.today())}_mut_sequences.fasta"
results = analyze_primers(msa_file, primers)

results.head()

In [ ]:
import matplotlib.patches as mpatches 

bins = range(0, max(results['mismatch_count']) + 2)

# plot mismatches per primer type 
fig, axes = plt.subplots(2,4, figsize = (14,5)) 
fig.tight_layout(h_pad = 4, w_pad = 4)
for ptype, ax in zip(['F3','F2','F1c','FL','B3','B2','B1c','BL'], axes.ravel()) :
    
    groups = results.loc[results.primer_type == ptype].groupby('record_description')['mismatch_count'].apply(list)
    
    # Plot each group
    ax.hist(groups, bins=bins, stacked=True, label=[x.replace("_","=") for x in groups.index], alpha=0.7,
           color =  sns.color_palette('Spectral_r',len(groups.index)))
    
    ax.set_title(ptype)
    ax.set_ylabel('Number of primers')
    ax.set_xlabel('Mismatches per target seq')
    #ax.legend(fontsize = 6)
    #ax.set_yscale('log')
   
fig.legend(title = "target sequence",
           handles = [mpatches.Patch(color = color, label = label) for color, label in zip(sns.color_palette('Spectral_r',len(groups.index)), groups.index)],
          bbox_to_anchor = (1.15,1))
fig.suptitle("Mismatches per primer colored by target seq", y = 1.1, fontsize = 14)

#fig.savefig(f"mismatches_per_primer_{date.today()}.pdf", format = "pdf")

In [ ]:
import scipy.stats as stats
# Function to calculate the mean and 95% CI
def mean_and_ci(group):
    mean = group.mean()
    sem = stats.sem(group)  # Standard error of the mean
    # 95% CI using the t-distribution
    ci_range = sem * stats.t.ppf((1 + 0.95) / 2, len(group) - 1)
    return mean, ci_range

# Group by 'record' and calculate statistics
summary = results.groupby('record')['mismatch_count'].apply(mean_and_ci)
summary
# Print the results
color =  sns.color_palette('Spectral_r',len(groups.index))
for i, (record, (mean, ci)) in enumerate(summary.items()):
    print(f"Record {record}: Mean mutations per primer = {mean:.2f}, 95% CI = ({mean-ci}, {mean+ci})")
    
    plt.bar(x = i, height = mean, yerr = ci, color =color[i])
    
plt.xticks(ticks = range(len(groups.index)), labels = groups.index, rotation = 90)

plt.title("Mean (95% CI) mutations per primer for different mutation rates per nucleotide")
plt.xlabel("Mutation rate per nucleotide (u)")
plt.ylabel("Mismatches per primer")

In [ ]:
results["mutation_rate"] = results.record_description.apply(lambda x: x.split("_")[1])
results["mutation_density"] = results.record_description.apply(lambda x: x.split("_")[-1])
results["primer_identity_to_target"] = [1 - (mm / len(t)) for mm, t in zip(results.mismatch_count,results.target_seq)]
results.sort_values(by = "mismatch_count", ascending = True)


In [ ]:
fig, (ax1,ax2) = plt.subplots(1,2, figsize = (12,4)) 

sns.heatmap(results.pivot_table(index = 'mutation_rate',columns = 'mutation_density',values = 'mismatch_count', aggfunc = 'mean'),
           cmap = "RdYlGn", square = True, linewidth = 0.1, linecolor = 'black', annot = True,
           cbar_kws = {'label':'mean mismatch count for primer:target pairs'}, ax = ax1)

ax1.set_title("Mean mismatches of all primer/target pairs\nfor different mutation rates and mutation densities")

sns.heatmap(results.pivot_table(index = 'mutation_rate',columns = 'mutation_density',values = 'primer_identity_to_target', aggfunc = 'mean'),
           cmap = "RdYlGn", square = True, linewidth = 0.1, linecolor = 'black', annot = True,
           cbar_kws = {'label':'mean identity for primer:target pairs'}, ax = ax2)

ax2.set_title("Mean identity of all primer/target pairs\nfor different mutation rates and mutation densities")



In [ ]:
fig, (ax1,ax2, ax3) = plt.subplots(1,3, figsize = (15,4)) 

sns.heatmap(results.pivot_table(index = 'mutation_rate',columns = 'mutation_density',values = 'mismatch_count', aggfunc = 'mean'),
           cmap = "RdYlGn", square = True, linewidth = 0.1, linecolor = 'black', annot = True,
           cbar_kws = {'label':'mean mismatch count for primer:target pairs'}, ax = ax1)

ax1.set_title("Mean mismatches of all primer/target pairs\nfor different mutation rates and mutation densities")

sns.heatmap(results.pivot_table(index = 'mutation_rate',columns = 'mutation_density',values = 'mismatch_count', aggfunc = 'max'),
           cmap = "RdYlGn", square = True, linewidth = 0.1, linecolor = 'black', annot = True,
           cbar_kws = {'label':'mean mismatch count for primer:target pairs'}, ax = ax2)

ax2.set_title("Max mismatches of all primer/target pairs\nfor different mutation rates and mutation densities")


sns.heatmap(results.pivot_table(index = 'mutation_rate',columns = 'mutation_density',values = 'primer_identity_to_target', aggfunc = 'mean'),
           cmap = "RdYlGn", square = True, linewidth = 0.1, linecolor = 'black', annot = True,
           cbar_kws = {'label':'mean identity for primer:target pairs'}, ax = ax3)

ax3.set_title("Mean identity of all primer/target pairs\nfor different mutation rates and mutation densities")


In [ ]:
sns.heatmap(results.pivot_table(index='primer_set', columns='record', values='mismatch_count', aggfunc='sum'))



In [ ]:
fig, ax = plt.subplots(1,1,figsize = (6,4))

by_pset = results.groupby(['primer_set', 'record_description'])['mismatch_count'].sum().reset_index()

bins = range(0, max(by_pset['mismatch_count']) + 2)

groups = by_pset.groupby('record_description')['mismatch_count'].apply(list)

ax.hist(groups, bins=bins, stacked=True, label=[x.replace("_","=") for x in groups.index], alpha=0.7,
           color =  sns.color_palette('Spectral_r',len(groups.index)))
    
ax.set_ylabel('Number of primer sets')
ax.set_xlabel('Mismatches per target seq')
#ax.set_yscale('log')

ax.set_title("Total mismatches per primer set\n 422 primer sets x 22 targets")

fig.legend(title = "target sequence",
           handles = [mpatches.Patch(color = color, label = label) for color, label in zip(sns.color_palette('Spectral_r',len(groups.index)), groups.index)],
          bbox_to_anchor = (1.2,1), fontsize = 8)

#fig.savefig(f"total_mismatches_{date.today()}.pdf", format = "pdf")

In [ ]:
import numpy as np
# Create a CDF plot for all targets combined
fig_all_cdf, ax_all_cdf = plt.subplots(1, 1, figsize=(4, 4))

# Flatten all mismatches across all targets
all_mismatches = np.concatenate(groups.values)
# Sort the mismatches
sorted_all_mismatches = np.sort(all_mismatches)
# Calculate the cumulative probability
y_vals_all = np.arange(1, len(sorted_all_mismatches) + 1) / len(sorted_all_mismatches)
# Plot the CDF
ax_all_cdf.plot(sorted_all_mismatches, y_vals_all, label='All targets combined', 
            linewidth=3, color='black')

ax_all_cdf.set_xlabel('Mismatches per LPS-target pair')
ax_all_cdf.set_ylabel('Cumulative probability')
ax_all_cdf.set_xlim(0, 25)
ax_all_cdf.set_ylim(0, 1.05)
ax_all_cdf.grid(True, linestyle='--', alpha=0.7)
ax_all_cdf.set_title('Cumulative Distribution Function of Mismatches Across All Targets')
ax_all_cdf.legend()

#fig_all_cdf.savefig(f"cumulative_total_mismatches_{date.today()}.pdf", format = "pdf")

#fig.savefig(f"total_mismatches_{date.today()}.pdf", format = "pdf")
pdf_path = "/Users/zlevine/Documents/Sabeti_Lab/LAMP_paper/figures_pdfs"
png_path = "/Users/zlevine/Documents/Sabeti_Lab/LAMP_paper/figures_pngs"

title = "Cumulative total mismatches"
plt.savefig(f"{pdf_path}/{title}.pdf", format = "pdf", bbox_inches = "tight")
plt.savefig(f"{png_path}/{title}.png", format = "png", bbox_inches = "tight")

In [ ]:
fig, ax = plt.subplots(1,1,figsize = (4,4))

by_pset = results.groupby(['primer_set', 'record_description'])['mismatch_count'].max().reset_index()

bins = range(0, max(by_pset['mismatch_count']) + 2)

groups = by_pset.groupby('record_description')['mismatch_count'].apply(list)

ax.hist(groups, bins=bins, stacked=True, label=[x.replace("_","=") for x in groups.index], alpha=0.7,
           color =  sns.color_palette('Spectral_r',len(groups.index)))
    
ax.set_ylabel('Number of primer-target pairs')
ax.set_xlabel('Maximum mismatches any primer in set')
#ax.set_yscale('log')

ax.set_title("Maximum mismatches any primer in set\n 422 primer sets x 22 targets")

fig.legend(title = "target sequence",
           handles = [mpatches.Patch(color = color, label = label) for color, label in zip(sns.color_palette('Spectral_r',len(groups.index)), groups.index)],
          bbox_to_anchor = (1.3,1), fontsize = 8)

#fig.savefig(f"total_mismatches_{date.today()}.pdf", format = "pdf")
pdf_path = "/Users/zlevine/Documents/Sabeti_Lab/LAMP_paper/figures_pdfs"
png_path = "/Users/zlevine/Documents/Sabeti_Lab/LAMP_paper/figures_pngs"

title = "Maximum mismatches any primer in set\n 422 primer sets x 22 targets"
plt.savefig(f"{pdf_path}/{title}.pdf", format = "pdf", bbox_inches = "tight")
plt.savefig(f"{png_path}/{title}.png", format = "png", bbox_inches = "tight")

In [ ]:
import numpy as np
# Create a CDF plot for all targets combined
fig_all_cdf, ax_all_cdf = plt.subplots(1, 1, figsize=(6, 4))

# Flatten all mismatches across all targets
all_mismatches = np.concatenate(groups.values)
# Sort the mismatches
sorted_all_mismatches = np.sort(all_mismatches)
# Calculate the cumulative probability
y_vals_all = np.arange(1, len(sorted_all_mismatches) + 1) / len(sorted_all_mismatches)
# Plot the CDF
ax_all_cdf.plot(sorted_all_mismatches, y_vals_all, label='All targets combined', 
            linewidth=3, color='black')

ax_all_cdf.set_xlabel('Maximum mismatches any primer in set')
ax_all_cdf.set_ylabel('Cumulative probability')
# ax_all_cdf.set_xlim(0, 30)
# ax_all_cdf.set_ylim(0, 1.05)
ax_all_cdf.grid(True, linestyle='--', alpha=0.7)
ax_all_cdf.set_title('Cumulative Distribution Function of Maximum mismatches any primer in set')
ax_all_cdf.legend()

#fig_all_cdf.savefig(f"cumulative_total_mismatches_{date.today()}.pdf", format = "pdf")

# Adding T7 and m13mp18 control region to existing targets 

In [ ]:
fasta_path = "/Users/zlevine/Documents/Sabeti_Lab/LAMP_ML/final_mutant_target_designs/2025-06-05_mut_sequences.fasta"
m13_path = "/Users/zlevine/Documents/Sabeti_Lab/LAMP_ML/final_mutant_target_designs/M13mp18-plus-minus-10.fasta"


In [ ]:
targ_alignment = AlignIO.read(fasta_path, "fasta") 
targ_alignment

In [ ]:
m13 = str(AlignIO.read(m13_path, "fasta")[0].seq)
print(len(m13))

In [ ]:
# add the T7 and m13 to each primer set 

T7 = 'TAATACGACTCACTATAG'

In [ ]:
# Create SeqRecord objects
seq_records = []
for record in targ_alignment:
    # Create a Seq object
    original_seq = record.seq
    record_id = record.id
    description = record.description
    ext_seq = Seq(T7 + m13 + str(original_seq))
    
    # Create a SeqRecord object
    seq_record = SeqRecord(ext_seq, id=record_id, description=description)

    # Add to list
    seq_records.append(seq_record)

# Optionally, write the SeqRecord objects to a file, e.g., in FASTA format
output_file = "/Users/zlevine/Documents/Sabeti_Lab/LAMP_ML/final_mutant_target_designs/2025-06-05_mut_sequences_w_t7_m13.fasta"
SeqIO.write(seq_records, output_file, "fasta")

In [ ]:
# also trim blocks to 300 only 
seq_records = []
for record in targ_alignment:
    # Create a Seq object
    original_seq = record.seq
    
    trim_seq = ""
    for block, bound in block_bounds.items() : 
        trim_seq += str(original_seq[bound[0]:bound[1]][0:300])
    record_id = record.id
    description = record.description
    ext_seq = Seq(T7 + m13 + str(trim_seq))
    
    # Create a SeqRecord object
    seq_record = SeqRecord(ext_seq, id=record_id, description=description)

    # Add to list
    seq_records.append(seq_record)

# Optionally, write the SeqRecord objects to a file, e.g., in FASTA format
output_file = "/Users/zlevine/Documents/Sabeti_Lab/LAMP_ML/final_mutant_target_designs/2025-06-05_mut_sequences_w_t7_m13_block_trim.fasta"
SeqIO.write(seq_records, output_file, "fasta")